In [6]:
import sys
import json
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import interpolate
from scipy.stats import gaussian_kde
from torch.utils.data import Dataset, DataLoader
import lightning as L

sys.path.insert(0, ".")
from x_jepa_arch import (
    JepaEncoder, JepaPredictor, GlucoEncoder, GluPredictor,
    sample_block_mask, ema_update, momentum_at, x_forward_loss,
)

CSV = "../../data/input/loop_ai_ready_joined2.csv"
WINDOW = 864
PATCH = 8
GRIDSIZE = 32
MAX_GAP_MIN = 60.0
N_CGM_PATCHES = WINDOW // PATCH
N_GLU_PATCHES = (GRIDSIZE // 8) ** 2
CACHE = Path("../../data/output/x_jepa_paired_train_w864.npz")
OUT_DIR = Path("../../jepa_nudes/x_jepa_encoder-864")

EPOCHS = 20
BATCH_SIZE = 256

In [4]:
def compute_2d_kde_grid(a, b, gridsize=64):
    kde = gaussian_kde(np.vstack([a, b]), bw_method="scott")
    a_min, a_max = np.percentile(a, [1, 99])
    b_min, b_max = np.percentile(b, [1, 99])
    Ag, Bg = np.meshgrid(np.linspace(a_min, a_max, gridsize), np.linspace(b_min, b_max, gridsize))
    Z = kde(np.vstack([Ag.ravel(), Bg.ravel()])).reshape(Ag.shape)
    return Z / (Z.max() + 1e-12)


def glucodensity(cgm_sequence, smoothing_factor=5.0, gridsize=64):
    cgm_sequence = np.asarray(cgm_sequence).flatten()
    t = np.arange(len(cgm_sequence)) * 5.0 / 60.0
    spline = interpolate.UnivariateSpline(t, cgm_sequence, s=smoothing_factor)
    g = spline(t)
    dg = spline.derivative(1)(t)
    ddg = spline.derivative(2)(t)
    return np.stack([
        compute_2d_kde_grid(g, dg, gridsize),
        compute_2d_kde_grid(g, ddg, gridsize),
        compute_2d_kde_grid(dg, ddg, gridsize),
    ], axis=-1)


def good_windows(glucose, ts, window, max_gap_min):
    step_min = np.diff(ts).astype("timedelta64[s]").astype(float) / 60.0
    for start in range(0, max(len(glucose) - window + 1, 0), window):
        w = glucose[start:start + window]
        gaps = step_min[start:start + window - 1]
        if np.isfinite(w).all() and (gaps <= max_gap_min).all():
            yield w


def build_paired(df):
    images, windows = [], []
    for _, g in df.groupby("User ID", sort=False):
        g = g.sort_values("Timestamp")
        glucose = g["Glucose (mg/dL)"].ffill().bfill().to_numpy(dtype=np.float64)
        ts = g["Timestamp"].to_numpy()
        for w in good_windows(glucose, ts, WINDOW, MAX_GAP_MIN):
            try:
                img = glucodensity(w, gridsize=GRIDSIZE).astype(np.float32)
            except np.linalg.LinAlgError:
                continue
            images.append(img)
            windows.append(w.astype(np.float32))
    return windows, images

In [3]:
class PairedDataset(Dataset):
    def __init__(self, windows, images):
        self.windows = torch.from_numpy(windows).float()
        self.images = torch.from_numpy(images).float()

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        return self.windows[i], self.images[i]


class XJepaLightning(L.LightningModule):
    def __init__(
        self,
        n_time_steps=288, patch_size=8, embed_dim=96,
        n_layers=3, n_heads=6, norm="instance",
        n_cgm_patches=None, n_glu_patches=16, gridsize=32,
        lr=1e-3, weight_decay=0.04,
        gluco_loss_weight=1.0, sigreg_weight=1.0,
        n_cgm_targets=4, n_glu_targets=8,
        min_block=3, max_block=6,
        ema_base=0.999, seed=0,
    ):
        super().__init__()
        n_cgm_patches = n_cgm_patches or n_time_steps // patch_size
        self.save_hyperparameters()
        self.rng = random.Random(seed)

        self.cgm_encoder = JepaEncoder(
            n_time_steps=n_time_steps, patch_size=patch_size, embed_dim=embed_dim,
            n_layers=n_layers, n_heads=n_heads, norm=norm,
        )
        self.cgm_encoder_ema = copy.deepcopy(self.cgm_encoder)
        for p in self.cgm_encoder_ema.parameters():
            p.requires_grad = False

        self.cgm_predictor = JepaPredictor(embed_dim=embed_dim, n_patches=n_cgm_patches)
        self.glu_encoder = GlucoEncoder(gridsize=gridsize, patch=8, embed_dim=embed_dim)
        self.glu_predictor = GluPredictor(num_gluco_patches=n_glu_patches, embed_dim=embed_dim)
        self.n_glu_patches = n_glu_patches

    def training_step(self, batch, batch_idx):
        glucose, gluco_img = batch
        h = self.hparams

        ctx, tgt = sample_block_mask(h.n_cgm_patches, h.n_cgm_targets, h.min_block, h.max_block, self.rng)
        glu_tgt = sorted(self.rng.sample(range(self.n_glu_patches), h.n_glu_targets))

        total, cgm_loss, gluco_loss, reg = x_forward_loss(
            self.cgm_encoder, self.cgm_encoder_ema, self.cgm_predictor,
            self.glu_encoder, self.glu_predictor,
            glucose, gluco_img,
            torch.tensor(ctx, device=self.device),
            torch.tensor(tgt, device=self.device),
            torch.tensor(glu_tgt, device=self.device),
            gluco_loss_weight=h.gluco_loss_weight,
            sigreg_weight=h.sigreg_weight,
        )
        self.log("train_total", total, prog_bar=True, on_epoch=True)
        self.log("train_cgm", cgm_loss, prog_bar=True, on_epoch=True)
        self.log("train_gluco", gluco_loss, prog_bar=True, on_epoch=True)
        self.log("train_sigreg", reg, on_epoch=True)
        return total

    def on_train_batch_end(self, outputs, batch, batch_idx):
        total_steps = self.trainer.estimated_stepping_batches
        momentum = momentum_at(self.global_step, total_steps, self.hparams.ema_base)
        ema_update(self.cgm_encoder_ema, self.cgm_encoder, momentum)

    def configure_optimizers(self):
        params = (
            list(self.cgm_encoder.parameters())
            + list(self.cgm_predictor.parameters())
            + list(self.glu_encoder.parameters())
            + list(self.glu_predictor.parameters())
        )
        return torch.optim.AdamW(params, lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)

In [7]:
if CACHE.exists():
    d = np.load(CACHE)
    windows, images = d["windows"], d["images"]
else:
    df = pd.read_csv(
        CSV,
        usecols=["User ID", "Timestamp", "Glucose (mg/dL)", "Recommended Split"],
        parse_dates=["Timestamp"],
    )
    df = df[df["Recommended Split"] == "train"]
    windows, images = build_paired(df)
    windows, images = np.stack(windows), np.stack(images)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE, windows=windows, images=images)

print(f"paired windows: {len(windows)}  glucose{windows.shape[1:]}  gluco{images.shape[1:]}")

loader = DataLoader(PairedDataset(windows, images), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/var/folders/sy/5plrpq6s4lxb_r_q_7kn2kvc0000gn/T/ipykernel_88783/3798094998.py:13: UserWarning: 
The maximal number of iterations maxit (set to 20 by the program)
allowed for finding a smoothing spline with fp=s has been reached: s
too small.
There is an approximation returned but the corresponding weighted sum
of squared residuals does not satisfy the condition abs(fp-s)/s < tol.
  spline = interpolate.UnivariateSpline(t, cgm_sequence, s=smoothing_factor)


paired windows: 6825  glucose(864,)  gluco(32, 32, 3)


In [8]:
model = XJepaLightning(n_time_steps=WINDOW, patch_size=PATCH, n_cgm_patches=N_CGM_PATCHES, n_glu_patches=N_GLU_PATCHES, gridsize=GRIDSIZE, min_block=13, max_block=27)
trainer = L.Trainer(max_epochs=EPOCHS, enable_checkpointing=False)
trainer.fit(model, loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ cgm_encoder     │ JepaEncoder   │  336 K │ train │     0 │
│ 1 │ cgm_encoder_ema │ JepaEncoder   │  336 K │ train │     0 │
│ 2 │ cgm_predictor   │ JepaPredictor │ 66.0 K │ train │     0 │
│ 3 │ glu_encoder     │ GlucoEncoder  │  354 K │ train │     0 │
│ 4 │ glu_predictor   │ GluPredictor  │ 37.8 K │ train │     0 │
└───┴─────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 794 K                                                                                            
Non-trainable params: 336 K                                                                                        
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.525                                                                      
Modules in train mode: 157                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/neprog/Desktop/kse/audio_dl/glucose-forecasting/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


/Users/neprog/Desktop/kse/audio_dl/glucose-forecasting/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/neprog/Desktop/kse/audio_dl/glucose-forecasting/.venv/lib/python3.14/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (26) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
`Trainer.fit` stopped: `max_epochs=20` reached.


In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.cgm_encoder.state_dict(), OUT_DIR / "encoder.pt")
config = {
    "n_time_steps": WINDOW, "patch_size": PATCH, "embed_dim": model.hparams.embed_dim,
    "n_layers": model.hparams.n_layers, "n_heads": model.hparams.n_heads, "norm": model.hparams.norm,
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(f"saved encoder + config to {OUT_DIR}")

saved encoder + config to ../../jepa_nudes/x_jepa_encoder-864
